## Validação de camada

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "data" / "dashboard").exists():
    ROOT = ROOT.parent

DASH_DIR = ROOT / "data" / "dashboard"

daily = pd.read_parquet(DASH_DIR / "dashboard_daily.parquet")
exceptions = pd.read_parquet(DASH_DIR / "dashboard_exceptions.parquet")
metrics = pd.read_parquet(DASH_DIR / "dashboard_model_metrics.parquet")
pricing = pd.read_parquet(DASH_DIR / "dashboard_pricing_scenarios.parquet")

print("daily", daily.shape)
print("exceptions", exceptions.shape)
print("metrics", metrics.shape)
print("pricing", pricing.shape)

daily (43860, 29)
exceptions (5111, 22)
metrics (23, 10)
pricing (240, 15)


In [3]:
assert not daily.empty
assert daily["date"].notna().all()
assert daily["product_id"].notna().all()
assert daily["region_id"].notna().all()
assert (daily["prediction"] >= 0).all()
assert (daily["upper_bound"] >= daily["lower_bound"]).all()
assert (daily["lower_bound"] >= 0).all()
assert daily.duplicated(["date", "product_id", "region_id"]).sum() == 0

if not metrics.empty:
    assert metrics["wape"].notna().all()
    assert metrics["mae"].notna().all()
    assert metrics["rmse"].notna().all()

print("Validação da camada de dashboard aprovada")

Validação da camada de dashboard aprovada


# View Ploty Pré Dashboard Power BI

## Dashboard Executivo

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

exec_daily = (
    daily.groupby("date", as_index=False)
    .agg(
        actual=("units_sold", "sum"),
        prediction=("prediction", "sum"),
        lower_bound=("lower_bound", "sum"),
        upper_bound=("upper_bound", "sum"),
        revenue_brl=("revenue_brl", "sum"),
        gross_margin_brl=("gross_margin_brl", "sum"),
        stockout_rate=("stockout_flag", "mean"),
    )
)

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Demanda real versus prevista",
        "Receita diária",
        "Margem diária",
        "Taxa de ruptura",
    ),
)

fig.add_trace(
    go.Scatter(
        x=exec_daily["date"],
        y=exec_daily["actual"],
        name="Real",
        line=dict(color="#1f77b4"),
    ), row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=exec_daily["date"],
        y=exec_daily["upper_bound"],
        name="Limite superior",
        line=dict(color="rgba(31,119,180,0.2)"),
        showlegend=False,
    ), row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=exec_daily["date"],
        y=exec_daily["lower_bound"],
        name="Limite inferior",
        fill="tonexty",
        fillcolor="rgba(31,119,180,0.12)",
        line=dict(color="rgba(31,119,180,0.2)"),
        showlegend=False,
    ), row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=exec_daily["date"],
        y=exec_daily["prediction"],
        name="Previsão",
        line=dict(color="#ff7f0e", width=2),
    ), row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=exec_daily["date"],
        y=exec_daily["revenue_brl"],
        name="Receita",
        line=dict(color="#2ca02c"),
        showlegend=False,
    ), row=1, col=2
)
fig.add_trace(
    go.Scatter(
        x=exec_daily["date"],
        y=exec_daily["gross_margin_brl"],
        name="Margem",
        line=dict(color="#9467bd"),
        showlegend=False,
    ), row=2, col=1
)
fig.add_trace(
    go.Scatter(
        x=exec_daily["date"],
        y=exec_daily["stockout_rate"],
        name="Ruptura",
        line=dict(color="#d62728"),
        showlegend=False,
    ), row=2, col=2
)

fig.update_layout(
    title="Demand Planning & Pricing — Preview Executivo",
    template="plotly_white",
    height=800,
    hovermode="x unified",
)
fig.update_yaxes(title_text="Unidades", row=1, col=1)
fig.update_yaxes(title_text="BRL", row=1, col=2)
fig.update_yaxes(title_text="BRL", row=2, col=1)
fig.update_yaxes(title_text="Taxa", tickformat=".1%", row=2, col=2)

preview_dir = ROOT / "dashboard_preview"
preview_dir.mkdir(exist_ok=True)
fig.write_html(preview_dir / "executive_demand.html", include_plotlyjs="cdn")
fig.show()

## Pag Operacional de Exceções

In [5]:
exception_counts = (
    exceptions.groupby(["exception_type", "severity"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

fig_exceptions = go.Figure()
for severity, color in {
    "high": "#d62728",
    "medium": "#ff7f0e",
    "low": "#2ca02c",
}.items():
    part = exception_counts[exception_counts["severity"] == severity]
    fig_exceptions.add_trace(
        go.Bar(
            x=part["exception_type"],
            y=part["count"],
            name=severity,
            marker_color=color,
        )
    )

fig_exceptions.update_layout(
    barmode="stack",
    title="Exceções operacionais por tipo e severidade",
    template="plotly_white",
    xaxis_title="Tipo de exceção",
    yaxis_title="Quantidade",
)
fig_exceptions.write_html(
    preview_dir / "operational_exceptions.html",
    include_plotlyjs="cdn",
)
fig_exceptions.show()

## Pag Avaliação do Modelo

In [7]:
metric_view = metrics.loc[
    metrics["segment_type"].isin(["global", "product", "region", "category"])
].copy()

metric_view["segment_label"] = (
    metric_view["segment_type"] + ": " + metric_view["segment_value"]
)

fig_model = go.Figure()
fig_model.add_trace(
    go.Bar(
        x=metric_view["segment_label"],
        y=metric_view["wape"],
        name="WAPE",
        marker_color="#1f77b4",
        hovertemplate="%{x}<br>WAPE=%{y:.2%}<extra></extra>",
    )
)
fig_model.update_layout(
    title="WAPE por segmento",
    template="plotly_white",
    xaxis_title="Segmento",
    yaxis_title="WAPE",
    xaxis_tickangle=-45,
)
fig_model.write_html(
    preview_dir / "model_evaluation.html",
    include_plotlyjs="cdn",
)
fig_model.show()